# 01 — Data Download and Feature Engineering

## Data source note
We use `yfinance` with `interval="1h"` to obtain ~2 years of intraday data.
Yahoo Finance restricts 5-minute bars to the last ~60 days; hourly bars are
available for up to 730 days. For a short validation window you can override
with `interval="5m", start="2024-11-01"`.

## Features
- **Market**: log-returns (1/3/6/12 bars), rolling volatility, relative range,
  volume ratio, time-of-day (sin/cos), Corwin-Schultz bid-ask spread estimate.
- **Order (synthetic)**: side (+1/-1), order_size_fraction, urgency.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # must come before pyplot import; remove for interactive use
import matplotlib.pyplot as plt

from data_loader import TICKERS, download_ohlcv
from features import compute_market_features, add_synthetic_orders, FEATURE_NAMES
from viz import FIGURES_DIR

In [2]:
# Download (uses parquet cache after first run).
# No explicit dates: defaults use today - 720 days, always within Yahoo's 730-day 1h limit.
data = download_ohlcv(TICKERS, interval='1h')
print(f'Tickers loaded: {list(data.keys())}')

[data] SPY: 3439 bars (2024-06-03 → 2026-05-22)


[data] QQQ: 3421 bars (2024-06-03 → 2026-05-22)


[data] AAPL: 3433 bars (2024-06-03 → 2026-05-22)


[data] MSFT: 3433 bars (2024-06-03 → 2026-05-22)


[data] NVDA: 3431 bars (2024-06-03 → 2026-05-22)


[data] AMZN: 3440 bars (2024-06-03 → 2026-05-22)


[data] GOOGL: 3438 bars (2024-06-03 → 2026-05-22)


[data] META: 3431 bars (2024-06-03 → 2026-05-22)


[data] JPM: 3441 bars (2024-06-03 → 2026-05-22)
Tickers loaded: ['SPY', 'QQQ', 'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'JPM']


In [3]:
# Inspect one ticker
spy = data['SPY']
print(spy.tail())
print(f'\nSPY: {len(spy):,} bars  |  date range: {spy.index[0].date()} → {spy.index[-1].date()}')

                                 open        high         low       close  \
2026-05-22 15:30:00+00:00  747.890015  748.039978  746.849976  747.380005   
2026-05-22 16:30:00+00:00  747.380005  748.770020  746.630005  748.734985   
2026-05-22 17:30:00+00:00  748.780029  748.940002  746.830017  747.309998   
2026-05-22 18:30:00+00:00  747.299988  747.469971  745.500000  745.580017   
2026-05-22 19:30:00+00:00  745.575012  746.210022  744.955017  745.590027   

                            volume  
2026-05-22 15:30:00+00:00  2711868  
2026-05-22 16:30:00+00:00  3141071  
2026-05-22 17:30:00+00:00  4062455  
2026-05-22 18:30:00+00:00  4164830  
2026-05-22 19:30:00+00:00  6896388  

SPY: 3,439 bars  |  date range: 2024-06-03 → 2026-05-22


In [4]:
# Compute market features for all tickers, each with its own seeded RNG
# (so synthetic-order randomness is independent of dict iteration order).
import hashlib

def _ticker_rng(ticker, base=42):
    h = hashlib.md5(f'{base}:{ticker}'.encode()).hexdigest()
    return np.random.default_rng(int(h[:8], 16) % (2**32))

all_features = {}
for ticker, df in data.items():
    mf = compute_market_features(df)
    feats = add_synthetic_orders(mf, rng=_ticker_rng(ticker))
    all_features[ticker] = feats
    print(f'{ticker}: {len(feats):,} rows after feature engineering')

SPY: 6,838 rows after feature engineering
QQQ: 6,802 rows after feature engineering
AAPL: 6,826 rows after feature engineering
MSFT: 6,826 rows after feature engineering
NVDA: 6,822 rows after feature engineering
AMZN: 6,840 rows after feature engineering
GOOGL: 6,836 rows after feature engineering
META: 6,822 rows after feature engineering
JPM: 6,842 rows after feature engineering


In [5]:
# Feature distributions for SPY
# FEATURE_NAMES has 13 entries; use a 4×4 grid (16 slots, last 3 hidden).
n_cols = 4
n_rows = (len(FEATURE_NAMES) + n_cols - 1) // n_cols  # ceil division → 4 rows
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
spy_feats = all_features['SPY']
for ax, col in zip(axes.flat, FEATURE_NAMES):
    spy_feats[col].hist(bins=50, ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
for ax in axes.flat[len(FEATURE_NAMES):]:
    ax.set_visible(False)
plt.suptitle('SPY — Feature Distributions', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


/var/folders/0s/_0lcfy6j7z37p9p_5x7nbyxc0000gn/T/ipykernel_16151/1025350682.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Corwin-Schultz spread over time (SPY)
fig, ax = plt.subplots(figsize=(12, 3))
spy_feats['spread_cs'].plot(ax=ax, lw=0.5, alpha=0.7)
ax.set_ylabel('Estimated bid-ask spread (Corwin-Schultz)')
ax.set_title('SPY — Corwin-Schultz Spread Estimate (1h bars)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'corwin_schultz_spy.png', dpi=150)
plt.show()

/var/folders/0s/_0lcfy6j7z37p9p_5x7nbyxc0000gn/T/ipykernel_16151/296992370.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
